# LSTM на словах (эксперимент)

Два способа текста в проекте:

1. **char** — следующий символ (`text_rnn.ipynb`, `train.py --level char`)
2. **word** — текст режется на слова, слово → id → эмбеддинг → следующий токен

Здесь разбираем второй путь. Код продакшена: `kizaru/tokenize.py`, `TextRNN`, `train.py --level word`, воркер `worker_lstm_word.py`, в боте модель `lstm-word`.

Тот же word-словарь потом можно скормить transformer/gpt (`train_lm.py --level word`). Пока Genius-корпус качается — полный трейн не гоняем, только короткий зонд на куске текста.


In [ ]:
from pathlib import Path
import sys

import torch
from torch import nn

ROOT = Path.cwd().resolve()
for candidate in (ROOT, ROOT.parent, ROOT.parent.parent):
    if (candidate / "core").is_dir():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.corpus.lyrics import get_batch, load_corpus
from core.text.rnn import TextRNN
from core.text.sample import evaluate, evaluate_words
from core.text.tokenize import (
    TOKEN_NL,
    TOKEN_UNK,
    detokenize,
    text_to_word_seq,
    tokenize,
)

print("ROOT", ROOT)


## 1. Токенизация

Не BPE и не BERT. Простой сплит: слова / числа / одиночная пунктуация, перевод строки = `<nl>`, редкие слова → `<unk>`.
Всё в нижнем регистре, чтобы словарь не раздувался от `Я` / `я`.


In [ ]:
sample = "Я еду по району, cousin.\nЭто мой дом."
tokens = tokenize(sample)
print(tokens)
print(detokenize(tokens))
assert TOKEN_NL in tokens


## 2. Словарь и последовательность id

`min_freq=3` в трейне отсекает разовые слова. Здесь для зонда ставим `1`, иначе на коротком тексте почти всё станет `<unk>`.


In [ ]:
corpus_path = ROOT / "data" / "corpus" / "kizaru_lyrics.txt"
if corpus_path.exists():
    full = load_corpus(corpus_path)
    # зонд: не весь Genius, только кусок — иначе ноутбук уедет в долгий прогон
    text = full[:120_000]
    print("corpus", corpus_path, "chars", len(full), "probe", len(text))
else:
    text = (sample + "\n") * 400
    print("fallback probe text", len(text))

sequence, token_to_idx, idx_to_token = text_to_word_seq(text, min_freq=1)
print("tokens", len(sequence), "vocab", len(token_to_idx))
print("special", TOKEN_UNK, token_to_idx[TOKEN_UNK], TOKEN_NL, token_to_idx[TOKEN_NL])
print("head", [idx_to_token[int(i)] for i in sequence[:20]])


## 3. Чем отличается от char

| | char | word |
|---|---|---|
| токен | символ | слово / знак / `<nl>` |
| vocab | сотни | тысячи+ |
| шаг генерации | буква | целое слово |
| чекпоинт | `kizaru.pt` | `kizaru_word.pt` |
| модель в боте | `lstm` | `lstm-word` |

Сеть та же: `TextRNN` (embedding → LSTM → linear). Меняется только алфавит: вместо букв — слова.
Обучение всё равно next-token + CrossEntropy, не RL. Генерация пошаговая через `step` / `evaluate_words`.


## 4. Короткий трейн-зонд

Маленькая сеть, мало шагов — только проверить, что градиент и сэмпл на словах живые. Боевой прогон: `python train.py --level word` из корня репо, когда Genius докачается.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)

model = TextRNN(
    input_size=len(token_to_idx),
    hidden_size=64,
    embedding_size=64,
    n_layers=2,
).to(device)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
crit = nn.CrossEntropyLoss()

seq_len = 32
steps = 80
for step in range(steps):
    model.train()
    train, target = get_batch(sequence, seq_len=seq_len, batch_size=8)
    train = train.permute(1, 0, 2).to(device)
    target = target.permute(1, 0, 2).to(device)
    hidden = model.init_hidden(8, device=device)
    out, _ = model(train, hidden)
    loss = crit(out.permute(1, 2, 0), target.squeeze(-1).permute(1, 0))
    opt.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if (step + 1) % 20 == 0:
        print(step + 1, float(loss))


In [ ]:
print(
    evaluate_words(
        model,
        token_to_idx,
        idx_to_token,
        start_text="Я",
        prediction_len=40,
        temp=0.8,
        device=device,
        max_lines=6,
    )
)


## 5. Рядом с char (если есть старый чекпоинт)

Не обязательно. Если лежит `checkpoints/kizaru.pt`, можно глянуть, как звучит char-генерация на том же старте. Word-чекпоинта до обучения ещё нет.


In [ ]:
from core.text.checkpoint import load_model

char_pt = ROOT / "checkpoints" / "kizaru.pt"
if char_pt.exists():
    char_model, char_to_idx, idx_to_char, meta = load_model(char_pt, device=device)
    print("char checkpoint", meta.get("arch"), "vocab", len(char_to_idx))
    print(
        evaluate(
            char_model,
            char_to_idx,
            idx_to_char,
            start_text="Я",
            prediction_len=120,
            temp=0.4,
            device=device,
            max_lines=4,
        )
    )
else:
    print("no", char_pt)


## 6. Боевые команды (из корня репо)

```bash
python train.py --level word --out checkpoints/kizaru_word.pt
python worker_lstm_word.py checkpoints/kizaru_word.pt --db data/bot.db
python generate.py --arch lstm-word --start "Я"
```

В Discord: `/ai text` → `lstm-word`.

Потом, тем же сплитом слов:

```bash
python train_lm.py --arch transformer --level word
python train_lm.py --arch gpt --level word
```

Не запускать полный трейн, пока Genius-пул не собран в `kizaru_lyrics.txt`.
